# POSRRI — Port Oil Spill Regulatory Readiness Index
## Chattogram Port, Bangladesh — full analysis pipeline

This notebook runs the complete, reproducible POSRRI analysis end to end:

| Step | Module | What it does |
|---|---|---|
| 0 | `src/structure.py` | Fixed hierarchy: 3 pillars → 10 domains → 50 indicators |
| 1 | `src/delphi.py` | Delphi consensus, I-CVI / S-CVI, round-to-round Cohen's κ |
| 2 | `src/ahp.py` | AHP weights by row geometric mean, consistency ratios, AIJ aggregation |
| 3 | `src/scoring.py` | Weighted domain, pillar and overall POSRRI on a 0–100 scale |
| 4 | `src/sensitivity.py` | AHP versus equal weights, Spearman rank correlation |
| 5 | `src/viz.py` | Six publication-grade figures (300 dpi PNG + vector PDF) |
| 6 | `src/survey_analysis.py` | Survey descriptives, Cronbach's alpha, Figure 7 |
| 7 | `src/report.py` | All result tables as CSV plus one combined Excel workbook |

**It runs identically in Google Colab and on a local machine.** The configuration
cell below detects Colab, installs dependencies, clones or updates the repository,
mounts Google Drive so results survive the runtime being recycled, and sets
`DATA_DIR` / `OUTPUT_DIR` accordingly.

### Before you press *Runtime → Run all*

Set **`USE_SYNTHETIC`** in the next cell:

* `True` — read the committed synthetic demonstration dataset in `synthetic/data/`.
  Use this to check the pipeline works, or to develop against realistic data.
* `False` — read real data from `data/raw/` (locally) or from your Drive folder
  (in Colab). **Real participant data must never be committed to the repository**;
  `data/raw/` is git-ignored for exactly this reason.

---
## Configuration

The only cell you normally need to edit. Everything downstream reads `PATHS`.

In [ ]:
# ============================ USER SETTINGS =================================
USE_SYNTHETIC = True          # True -> synthetic/data ; False -> real data
DRIVE_SUBDIR = "POSRRI"       # folder inside My Drive used for real data + outputs
REPO_URL = "https://github.com/SKPrince1911/posrri-analysis.git"
REPO_BRANCH = "claude/posrri-pipeline-setup-0jn6k7"
# ============================================================================

import os
import subprocess
import sys
from pathlib import Path

try:                                    # Colab exposes this package
    import google.colab            # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE_ROOT = None
DRIVE_AVAILABLE = False

if IN_COLAB:
    print("Google Colab detected.")

    # 1. Dependencies (quiet; most are already present in the Colab image).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "numpy", "scipy", "matplotlib", "openpyxl"],
                   check=False)

    # 2. Clone the repository, or fast-forward it if it is already there.
    REPO_DIR = Path("/content/posrri-analysis")
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH],
                       check=False)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL,
                        str(REPO_DIR)], check=False)

    # 3. Mount Drive so tables and figures outlive the runtime.
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        (DRIVE_ROOT / "data" / "raw").mkdir(parents=True, exist_ok=True)
        OUTPUT_DIR = DRIVE_ROOT / "outputs"
        DRIVE_AVAILABLE = True
        print(f"Drive mounted. Outputs -> {OUTPUT_DIR}")
    except Exception as exc:                        # noqa: BLE001
        print(f"Drive not mounted ({exc}). Outputs stay in the Colab runtime "
              f"and will be lost when it recycles.")
        OUTPUT_DIR = REPO_DIR / "outputs"
else:
    # Local / CI: the notebook lives in <repo>/notebooks.
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / "config.py").exists() and (REPO_DIR.parent / "config.py").exists():
        REPO_DIR = REPO_DIR.parent
    OUTPUT_DIR = REPO_DIR / "outputs"
    print(f"Local run. Repository root: {REPO_DIR}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import config                                        # noqa: E402

# Where the inputs come from.
if USE_SYNTHETIC:
    DATA_DIR = REPO_DIR / "synthetic" / "data"
elif IN_COLAB and DRIVE_AVAILABLE:
    DATA_DIR = DRIVE_ROOT / "data" / "raw"           # real data lives in Drive
else:
    DATA_DIR = REPO_DIR / "data" / "raw"             # real data, git-ignored

PATHS = config.resolve_paths(use_synthetic=USE_SYNTHETIC,
                             data_dir=DATA_DIR, output_dir=OUTPUT_DIR)
FIGURE_DIR, TABLE_DIR = PATHS["figure_dir"], PATHS["table_dir"]

print(f"\nUSE_SYNTHETIC = {USE_SYNTHETIC}")
print(f"DATA_DIR      = {PATHS['data_dir']}")
print(f"OUTPUT_DIR    = {PATHS['output_dir']}")
print(f"  figures     -> {FIGURE_DIR}")
print(f"  tables      -> {TABLE_DIR}")

### Load the data

If the synthetic dataset is missing (a fresh clone regenerates it in seconds),
it is rebuilt here from the fixed master seed, so the numbers below are
identical on every machine.

In [ ]:
import numpy as np
import pandas as pd

from src import structure as st
from src.ahp import run_ahp
from src.delphi import run_delphi
from src.report import collect_tables, export_tables
from src.scoring import benchmark_matrix, score_index
from src.sensitivity import run_sensitivity
from src import viz

data_dir = PATHS["data_dir"]
needed = ["delphi_ratings.csv", "ahp_pairwise.csv", "scores.csv",
          "survey.csv", "benchmark.csv"]
absent = [f for f in needed if not (data_dir / f).exists()]

if absent and USE_SYNTHETIC:
    print(f"Regenerating synthetic data (missing: {absent})")
    from synthetic.generate_synthetic import generate_all
    generate_all(seed=config.RANDOM_SEED, outdir=data_dir)
elif absent:
    raise FileNotFoundError(
        f"Missing real data file(s) in {data_dir}: {absent}\n"
        f"Copy the templates from data/templates/ and fill them in, "
        f"or set USE_SYNTHETIC = True."
    )

delphi_raw    = pd.read_csv(data_dir / "delphi_ratings.csv")
ahp_raw       = pd.read_csv(data_dir / "ahp_pairwise.csv")
scores_raw    = pd.read_csv(data_dir / "scores.csv")
survey_raw    = pd.read_csv(data_dir / "survey.csv")
benchmark_raw = pd.read_csv(data_dir / "benchmark.csv")

print(f"delphi_ratings : {delphi_raw.shape[0]:>5} rows  "
      f"({delphi_raw['expert_id'].nunique()} experts x "
      f"{delphi_raw['indicator_code'].nunique()} indicators x "
      f"{delphi_raw['round'].nunique()} rounds)")
print(f"ahp_pairwise   : {ahp_raw.shape[0]:>5} rows  "
      f"({ahp_raw['expert_id'].nunique()} experts)")
print(f"scores         : {scores_raw.shape[0]:>5} rows")
print(f"survey         : {survey_raw.shape[0]:>5} rows")
print(f"benchmark      : {benchmark_raw.shape[0]:>5} rows  "
      f"({benchmark_raw['port'].nunique()} ports)")

---
## Step 0 — Index structure

The hierarchy is fixed in code, not inferred from the data, so every stage
shares one definition and the structure cannot drift between the Delphi, the
AHP and the scoring. `validate_structure()` re-checks the invariants
(3 pillars, 10 domains partitioned across them, 5 indicators per domain).

In [ ]:
ok, problems = st.validate_structure()
print(f"Structure valid: {ok}" + ("" if ok else f"  problems: {problems}"))
print(f"{st.N_PILLARS} pillars | {st.N_DOMAINS} domains | {st.N_INDICATORS} indicators\n")

for pillar, domains in st.PILLAR_DOMAINS.items():
    print(f"Pillar {pillar}: {st.PILLARS[pillar]}")
    for dom in domains:
        print(f"    {dom:<4} {st.DOMAINS[dom]}")

st.structure_frame().head(8)

---
## Step 1 — Delphi consensus and content validity

For each indicator and round the module computes the median relevance, the IQR,
the percentage of experts rating 7–9, and the consensus flag defined by the
protocol:

> **consensus** = (≥ 80 % of experts rate 7–9) **or** (median ≥ 7 **and** IQR ≤ 2)

Content validity follows Zamanzadeh et al. (2015): **I-CVI** is the proportion
rating 7–9, with items retained at I-CVI ≥ 0.78; the chance-corrected modified
kappa κ\* is reported alongside; **S-CVI/Ave** is the mean I-CVI, reported
against a target of ≥ 0.90.

Stability between rounds is measured with **Cohen's κ** on the retain/drop
decisions, following the recommendation (Hohmann et al. 2025) that Delphi
studies report a stability criterion as well as a consensus criterion.

In [ ]:
delphi_res = run_delphi(delphi_raw)
s = delphi_res["summary"]
last_round = s["rounds"][-1]

print("Delphi summary")
print("-" * 66)
print(f"  Experts                       {s['n_experts']}")
print(f"  Rounds                        {s['rounds']}")
for r in s["rounds"]:
    print(f"  Round {r}: consensus            {s[f'n_consensus_r{r}']:>2} / {s['n_indicators']}"
          f"   retained {s[f'n_retained_r{r}']:>2}"
          f"   S-CVI/Ave {s[f'scvi_ave_r{r}']:.3f}")
print(f"  S-CVI/Ave target ({config.SCVI_TARGET})        "
      f"{'MET' if s['scvi_target_met'] else 'NOT MET'}")
print(f"  S-CVI/Ave over retained items {s['scvi_ave_retained']:.3f}")
print(f"  Cohen's kappa (round 1 vs {last_round})    {s['cohens_kappa']:.3f}"
      f"   (po = {s['kappa_po']:.3f}, pe = {s['kappa_pe']:.3f})")
print(f"  Decisions changed             {s['n_decisions_changed']}")

print("\nRound-1 vs round-2 decision cross-tabulation:")
print(delphi_res["kappa"]["table"].to_string())

print(f"\nIndicators failing the I-CVI cut-off in round {last_round} "
      f"({len(delphi_res['dropped'])}):")
delphi_res["dropped"]

**Note on scope.** The Delphi is the instrument-development stage: it reports which
indicators the panel endorsed. This demonstration then scores the **full 50-indicator
set** so that the weight and score tables are complete. In a live application you
would carry the retained set forward and re-elicit weights over it; the retained and
dropped lists are exported for exactly that purpose.

In [ ]:
cols = ["indicator_code", "domain_code",
        f"median_relevance_r{last_round}", f"iqr_relevance_r{last_round}",
        f"pct_rating_7_9_r{last_round}", f"I_CVI_r{last_round}",
        f"kappa_star_r{last_round}", f"consensus_r{last_round}", "decision"]
delphi_res["per_indicator"][cols].round(3).head(12)

---
## Step 2 — AHP weights

Pairwise judgements arrive in long form (`item_i`, `item_j`, `saaty_value`), with
fractions such as `1/3` accepted as text and reciprocals filled automatically.
For each matrix:

* priority weights come from the **normalised row geometric mean**;
* $\lambda_{max} = \mathrm{mean}\big((A\mathbf{w}) / \mathbf{w}\big)$,
  $CI = (\lambda_{max} - n) / (n - 1)$, $CR = CI / RI(n)$ with Saaty's Random Index;
* $CR \ge 0.10$ is flagged.

Experts are combined by **Aggregation of Individual Judgements (AIJ)** — the
cell-wise geometric mean of the judgements, taken *before* the group priority
vector is derived, which is the only rule that preserves the reciprocal property.

The global weight of an indicator is
`pillar_w × domain_w_within_pillar × indicator_w_within_domain`, normalised to
sum to 1 over all 50 indicators.

In [ ]:
ahp_res = run_ahp(ahp_raw)
a = ahp_res["summary"]

print("AHP summary")
print("-" * 66)
print(f"  Experts                       {a['n_experts']}")
print(f"  Matrices in the hierarchy     {a['n_matrices']}  (1 pillar + 3 domain + 10 indicator)")
print(f"  Consistency checks            {a['n_consistency_rows']}  "
      f"(each expert per matrix, plus the AIJ group matrix)")
print(f"  Matrices with CR >= {a['cr_threshold']}      {a['n_inconsistent']}")
print(f"  Maximum CR observed           {a['max_CR']:.4f}")
print(f"  Global weights sum to         {a['global_weight_sum']:.10f}")

print("\nPillar weights")
print(ahp_res["pillar_weights"].round(4).to_string(index=False))

print("\nDomain weights")
print(ahp_res["domain_weights"][
    ["domain_code", "domain_name", "pillar_code",
     "domain_weight_within_pillar", "domain_weight_global"]
].round(4).to_string(index=False))

In [ ]:
print("Ten most heavily weighted indicators")
display_cols = ["indicator_code", "indicator_name", "domain_code", "global_weight"]
ahp_res["weights"].nlargest(10, "global_weight")[display_cols].round(4)

In [ ]:
print("Consistency ratios by matrix (AIJ group matrices)")
group_cr = ahp_res["consistency"].query("expert_id == 'AIJ_GROUP'")
group_cr[["level", "group", "n_items", "lambda_max", "CI", "RI", "CR", "consistent"]].round(4)

---
## Step 3 — Scoring

Each indicator carries an adjudicated `final_score` on the 0–3 rubric
(0 absent, 1 partial or ambiguous, 2 defined, 3 defined and verified). For any group
$g$ of indicators:

$$\mathrm{score}(g) = 100 \times \frac{\sum_{i \in g} w_i s_i}{3 \sum_{i \in g} w_i}$$

Dividing by the group's own weight mass makes domain scores comparable with one
another regardless of how much of the global weight each domain holds. Because
the 50 global weights sum to 1, the overall POSRRI is simply
$100 \times \sum_i w_i s_i / 3$.

In [ ]:
score_res = score_index(scores_raw, ahp_res["weights"])
o = score_res["overall"]

print("=" * 66)
print(f"  POSRRI = {o['POSRRI_0_100']:.2f} / 100     ({o['band']})")
print("=" * 66)
print(f"  Unweighted equivalent          {o['unweighted_POSRRI_0_100']:.2f}")
print(f"  Mean indicator score (0-3)     {o['mean_final_score_0_3']:.2f}")
print(f"  Strongest domain               {o['strongest_domain']}  "
      f"{st.DOMAINS[o['strongest_domain']]}")
print(f"  Weakest domain                 {o['weakest_domain']}  "
      f"{st.DOMAINS[o['weakest_domain']]}")
print(f"  Mean implementation gap        {o['mean_implementation_gap']:+.2f} rubric points")

print("\nPillar scores")
print(score_res["pillars"][["pillar_code", "pillar_name", "weight",
                            "score_0_100", "band"]].round(3).to_string(index=False))

print("\nDomain scores")
score_res["domains"][["domain_code", "domain_name", "weight", "mean_final_score",
                      "score_0_100", "band", "rank"]].round(3)

In [ ]:
print("Indicators contributing most to the overall shortfall")
print("(large weight combined with a low score = highest-priority remediation)")
cols = ["indicator_code", "indicator_name", "domain_code", "final_score",
        "weight", "shortfall_share_pct"]
score_res["indicators"].nlargest(10, "shortfall_share_pct")[cols].round(4)

---
## Step 4 — Sensitivity to the weighting scheme

A composite index is only as credible as its weights, so the aggregation is
re-run with every indicator weighted 1/50 (the "no information" null) and the
two domain **rankings** — the thing that actually drives policy priorities — are
compared with Spearman's ρ (Kendall's τ-b reported as a tie-robust companion).

In [ ]:
sens_res = run_sensitivity(scores_raw, ahp_res["weights"])
sm = sens_res["summary"]

print("Sensitivity: AHP weights versus equal weights")
print("-" * 66)
print(f"  POSRRI, AHP weights           {sm['POSRRI_ahp']:.2f}")
print(f"  POSRRI, equal weights         {sm['POSRRI_equal']:.2f}")
print(f"  Difference                    {sm['POSRRI_delta']:+.2f} points "
      f"(band unchanged: {sens_res['overall']['band_unchanged']})")
print(f"  Spearman rho                  {sm['spearman_rho']:.3f}  "
      f"(p = {sm['spearman_p']:.3g})")
print(f"  Kendall tau-b                 {sm['kendall_tau']:.3f}  "
      f"(p = {sm['kendall_p']:.3g})")
print(f"  Domains changing rank         {sm['n_rank_changes']} of {st.N_DOMAINS}"
      f"  (largest shift {sm['max_rank_shift']} place(s))")
print(f"  Largest score change          {sm['domain_with_max_delta']}, "
      f"{sm['max_abs_domain_delta']:.2f} points")
print(f"  Top three, AHP                {sm['top3_ahp']}")
print(f"  Top three, equal              {sm['top3_equal']}")
print(f"  Conclusions robust            {sm['conclusion_robust']}")

sens_res["domains"][["domain_code", "domain_name", "score_ahp", "score_equal",
                     "delta", "rank_ahp", "rank_equal", "rank_shift"]].round(2)

---
## Step 5 — Figures

Six figures, each written as a 300 dpi PNG (review and submission) and a vector
PDF (typesetting), all on the colourblind-safe Okabe-Ito palette with a second
encoding channel — line style, marker, or a printed value — so nothing depends
on colour alone.

In [ ]:
cpa_domain_scores = score_res["domains"].set_index("domain_code")["score_0_100"]
bench_0_100 = benchmark_matrix(benchmark_raw, cpa_domain_scores=cpa_domain_scores)

figures = viz.make_all_figures(
    scoring_result=score_res,
    delphi_result=delphi_res,
    sensitivity_result=sens_res,
    benchmark=benchmark_raw,
    ahp_domain_weights=ahp_res["domain_weights"],
    figure_dir=FIGURE_DIR,
)

for stem, paths in figures.items():
    print(f"  {stem:<28} " + ", ".join(p.name for p in paths))

print("\nDomain scores on the 0-100 scale, by port:")
bench_0_100.round(1)

In [ ]:
# Display the figures inline. Colab and Jupyter render them; if the notebook is
# executed as a plain script the paths are printed instead.
try:
    from IPython.display import Image, display
    INLINE = True
except ImportError:
    INLINE = False

for stem in viz.FIGURE_STEMS:
    png = FIGURE_DIR / f"{stem}.png"
    if not png.exists():
        continue
    print(f"\n{'=' * 70}\n{stem}\n{'=' * 70}")
    if INLINE:
        display(Image(filename=str(png), width=760))
    else:
        print(f"  {png}")

---
## Step 5b — Stakeholder survey

The survey is ingested separately from the index, because it arrives in a
different shape: a raw **Google Forms** export whose headers are full question
sentences and whose answers are category labels.

`src/survey_ingest.py` handles that conversion. It drops the `Timestamp` and
consent (Q0) columns — after checking consent and reporting any row that did not
answer "Yes" — maps the remaining columns to `q1`…`q15` **by position** (never by
header text, which changes whenever the form is edited), assigns
`AGT-001`, `AGT-002`, … in ascending timestamp order, and recodes the categorical
answers using the explicit tables at the top of that module.

> **Where the cleaned file goes.** The ingest writes to `data/raw/survey.csv` by
> default, because a real export contains participant responses and `data/raw/`
> is git-ignored. It never writes into `synthetic/data/`.

The synthetic `survey.csv` used below is itself produced by running this same
ingest over a synthetic raw export, so there is one implementation of the survey
coding rather than two that could drift apart.

In [ ]:
from src.survey_analysis import run_survey_analysis
from src.survey_ingest import load_survey_export, read_survey

# With real data you would start from the raw export:
#     survey = load_survey_export("MyDrive/POSRRI/data/raw/forms_export.csv")
# The synthetic dataset already carries both the raw export and the cleaned file,
# so re-run the ingest here to show what it reports.
export_path = data_dir / "survey_export.csv"
if export_path.exists():
    survey = load_survey_export(export_path, write=False)      # prints its summary
else:
    survey = read_survey(data_dir / "survey.csv")

survey.head(8)

In [ ]:
survey_res = run_survey_analysis(survey, figure_dir=FIGURE_DIR)
sv = survey_res["summary"]
a = survey_res["alpha"]

print("Survey analysis")
print("-" * 70)
print(f"  Respondents                   {sv['n_respondents']}")
print(f"  Items                         {sv['n_items']} "
      f"({sv['n_likert_items']} Likert, {sv['n_categorical_items']} coded, "
      f"{sv['n_text_items']} free text)")
print(f"  Missing coded values          {sv['total_missing']}")
print(f"  Cronbach's alpha (7 items)    {a['alpha']:.3f}  ({a['interpretation']})")
print(f"    complete cases              {a['n_respondents']} "
      f"({a['n_excluded_incomplete']} excluded by listwise deletion)")
print(f"  Highest / lowest rated item   {sv['highest_rated_item']} / {sv['lowest_rated_item']}")

print("\nItem statistics for the confidence scale "
      "(a low item-total r, or an alpha_if_deleted above the overall alpha, "
      "flags an item that does not belong):")
print(a["item_statistics"][["item", "label", "mean", "sd", "item_total_r",
                            "alpha_if_deleted"]].round(3).to_string(index=False))

survey_res["likert"][["item", "label", "n", "mean", "sd", "median",
                      "pct_1_2", "pct_4_5"]].round(2)


In [ ]:
print("Frequency tables for the coded categorical items:")
print(survey_res["frequencies"][["item", "category", "count", "pct_of_valid",
                                 "n_missing"]].round(1).to_string(index=False))

print("\nFree-text items:")
print(survey_res["text"][["item", "n_answered", "n_blank", "n_distinct"]]
      .to_string(index=False))

for path in survey_res["figure"]:
    print(f"\nwrote {path.name}")
png = FIGURE_DIR / "fig7_survey_likert.png"
if INLINE and png.exists():
    display(Image(filename=str(png), width=820))


---
## Step 6 — Export the result tables

Every table is written as a CSV, and all of them together as a single
multi-sheet Excel workbook (`posrri_results.xlsx`) for circulation to
co-authors and for the journal's supplementary material.

In [ ]:
tables = collect_tables(delphi_res, ahp_res, score_res, sens_res,
                        benchmark_0_100=bench_0_100,
                        survey_result=survey_res)
written = export_tables(tables, table_dir=TABLE_DIR, verbose=True)

print(f"\n{len(tables)} tables written to {TABLE_DIR}")
print(f"Combined workbook: {written['__workbook__']}")

tables["00_summary"]

---
## Reproducibility notes

* **Determinism.** The synthetic dataset is generated from a single master seed
  (`config.RANDOM_SEED`), split into independent `SeedSequence` streams. Re-running
  this notebook on any machine reproduces every number above exactly.
* **No hidden state.** Every threshold (I-CVI 0.78, S-CVI target 0.90, consensus
  rule, CR 0.10, the 0–3 rubric) lives in `config.py`, not scattered through the
  analysis code.
* **Validation.** `python tests/validate_pipeline.py` re-runs this whole pipeline
  and prints a PASS/FAIL checklist covering consistency ratios, weight
  normalisation, score ranges, Delphi outputs and figure generation.

## Working with real data

1. Copy the header-only templates from `data/templates/` into `data/raw/`
   (locally) or into `MyDrive/POSRRI/data/raw/` (Colab) and fill them in. The
   schemas are documented in `data/templates/README.md`.
2. Set `USE_SYNTHETIC = False` in the configuration cell.
3. *Runtime → Run all.* Loaders validate columns, code coverage and value ranges,
   and fail with an explicit message rather than producing a silently wrong index.

**`data/raw/` is git-ignored.** Real participant data must never be committed.

## Method references

* Saaty, T.L. (1980) *The Analytic Hierarchy Process.* McGraw-Hill — AHP,
  the 1–9 scale, the Random Index and the consistency ratio.
* Zamanzadeh, V. et al. (2015) 'Design and implementation content validity study',
  *Journal of Caring Sciences* 4(2), 165–178 — I-CVI, S-CVI/Ave, modified kappa.
* Hohmann, E. et al. (2025) — Delphi methodology: reporting both a consensus
  criterion and a stability criterion across rounds.
* Okabe, M. & Ito, K. (2008) 'Color universal design' — the colourblind-safe palette.